# Environment Setup

This notebook walks through installing and verifying `oneccl_bindings_for_pytorch`
on a CRI inference node. Run this before any other notebook.

## Step 1 -- Load Intel oneAPI Modules

In [ ]:
# Check current module environment
import subprocess
result = subprocess.run("module list 2>&1", shell=True, capture_output=True, text=True)
print(result.stdout or result.stderr)

# If Intel MPI not loaded, uncomment:
# !module load intel/mpi intel/oneapi/pytorch

## Step 2 -- Install oneCCL PyTorch Bindings

In [ ]:
# Install from Intel's PyPI channel
# Pin to the version matching your PyTorch build
!pip install oneccl_bind_pt --extra-index-url https://pytorch-extension.intel.com/release-whl/stable/xpu/us/ 2>&1 | tail -5

## Step 3 -- Full Verification

In [ ]:
%%writefile /tmp/ccl_verify.py
import os
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_LOG_LEVEL"] = "warn"

dist.init_process_group(backend="ccl")

rank = dist.get_rank()
size = dist.get_world_size()
device = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

t = torch.tensor([float(rank)], device=device)
dist.all_reduce(t)
expected = sum(range(size))

status = "PASS" if abs(t.item() - expected) < 0.01 else "FAIL"
print(f"[rank {rank}/{size}] device={device}  allreduce={t.item():.0f}  expected={expected}  {status}")

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_verify.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    print("\nTroubleshooting:")
    print("  - Ensure `module load intel/mpi` was run before launching Jupyter")
    print("  - Check: echo $I_MPI_ROOT")
    print("  - Check: python -c 'import oneccl_bindings_for_pytorch'")

## Step 4 -- NUMA Topology Inspection

In [ ]:
# This is the most important pre-flight check for CRI nodes
# You want GPUs distributed evenly across NUMA domains

import subprocess

print("=== NUMA Hardware ===")
r = subprocess.run("numactl --hardware", shell=True, capture_output=True, text=True)
print(r.stdout)

print("=== XPU Devices ===")
r2 = subprocess.run(
    "python -c \""
    "import torch; "
    "[print(f'xpu:{i} -> {torch.xpu.get_device_properties(i).name}') "
    "for i in range(torch.xpu.device_count())]\"",
    shell=True, capture_output=True, text=True
)
print(r2.stdout)

If your environment passes all checks above, proceed to
[Allreduce — The TP Decode Hot Path](03a_allreduce_walkthrough.ipynb).

---

## How to Run These Notebooks

All notebooks use a **launcher pattern**: each hands-on cell writes a Python script to
`/tmp/` and runs it via `mpirun` as a subprocess. This means:

- You run the notebook from a **single-rank Jupyter kernel** (one Python process)
- The actual multi-GPU collective work happens in a separate `mpirun -n 4` subprocess
- Output from all 4 ranks appears in the cell output below the launcher cell

**You do not need to launch Jupyter with mpirun.** Just start a normal single-kernel
Jupyter session. Every notebook cell that needs multiple ranks will launch them itself.

### Step-by-step

```
1. SSH into your CRI node (or open a JupyterHub session)

2. Load Intel MPI and oneAPI environment:
      source /opt/intel/oneapi/setvars.sh
   or:
      module load intel/mpi intel/oneapi/pytorch

3. Launch JupyterLab (single kernel, no mpirun needed):
      jupyter lab --no-browser --port=8888
   then open the URL in your browser (or use VS Code's Jupyter extension)

4. Open any notebook under notebooks/
   Start with 02_environment_setup.ipynb (this notebook) to verify your stack.

5. Run cells top-to-bottom. Cells that start with %%writefile write a script to /tmp/.
   The next cell runs it with mpirun. You will see output from all 4 ranks.
```

### What the launcher looks like

```python
%%writefile /tmp/my_script.py   ← writes the multi-rank script
import torch.distributed as dist
...

# Then in the next cell:
result = subprocess.run("mpirun -n 4 -ppn 4 python /tmp/my_script.py", ...)
print(result.stdout)            ← shows output from all 4 ranks
```

### If mpirun isn't found

Run `which mpirun` in a terminal. If it's missing, you need to source the Intel oneAPI
environment first (`source /opt/intel/oneapi/setvars.sh`). The environment check cell
at the top of each notebook catches this before you reach any benchmark code.